In [1]:
from datasets import load_dataset

dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy","clean",split="validation")
dataset

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

Dataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_rows: 73
})

In [ ]:
# Inspecting one trace

from IPython.display import Audio

sample = dataset[2]

print(sample["text"])
Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"]) # array = amplitude, sr how many sample per second to play

HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND


In [4]:
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-base-100h")

# Since the pipeline consumes any dictionary that we pass it 
# (meaning it cannot be re-used), we’ll pass a copy of the data.

#The audio has to be decode first

audio = sample["audio"].get_all_samples()

audio_array = audio.data[0].numpy() # first channel
sampling_rate = audio.sample_rate

pipe({"raw":audio_array, "sampling_rate": sampling_rate})

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'text': 'HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAUS AND ROSE BEEF LOOMING BEFORE US SIMALYIS DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND'}

For comparison here are the original and the transcription reported:

*Original sample*   HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND

*Transcription*   HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAUS AND ROSE BEEF LOOMING BEFORE US SIMALYIS DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND


This highlights the shortcoming of a CTC model, which is essentially ‘acoustic-only’: it consists of an encoder which forms hidden-state representations from the audio inputs, and a linear layer which maps the hidden-states to characters:

This means that the system almost entirely bases its prediction on the acoustic input it was given (the phonetic sounds of the audio), and so has a tendency to transcribe the audio in a phonetic way (e.g. CHRISTMAUS). It gives less importance to the language modelling context of previous and successive letters, and so is prone to phonetic spelling errors.

In [ ]:
# Moving to Seq2seq

import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

pipe = pipeline("automatic-speech-recognition", model="openai/whisper-base", device=device)

pipe(sample["audio"],max_new_tokens=256) # Whisper captures punctutation and casing correctly.

cpu


config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  290MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more 

{'text': ' He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similarly is drawn from eating and its results occur most readily to the mind.'}

In [17]:
# multilingual test 

ml_dataset= load_dataset("facebook/multilingual_librispeech", "italian", split="dev",streaming=True)

sample=next(iter(ml_dataset))

print(sample["audio"])
Audio(sample["audio"]["array"],rate=sample["audio"]["sampling_rate"])


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

In [18]:
# pipe(sample["audio"].copy(), max_new_tokens=256, generate_kwargs={"task": "transcribe"})


# Again before passing, we need to decode 

audio = sample["audio"].get_all_samples()

audio_array = audio.data[0].numpy() # first channel
sampling_rate = audio.sample_rate

pipe({"raw":audio_array, "sampling_rate": sampling_rate}, generate_kwargs={"task":"translate"}) # kwarge allow translation in english

{'text': ' So much, give it to me, be the cousin of God, the mother still judge where she is. So much pleasure, Lord, to be able to rise up to this joy and joy.'}

---

In [20]:
# Long-from transcription

import numpy as  np 

target_length_in_m = 5 

# converting minutes to seconds to n.samples
sampling_rate = pipe.feature_extractor.sampling_rate
target_length_in_samples = target_length_in_m * 60 * sampling_rate

# iterate over streaming dataset, concatenating samples unitle target is achieve
long_audio = []
for sample in ml_dataset:
    long_audio.extend(sample["audio"]["array"])
    if len(long_audio) > target_length_in_samples:
        break

long_audio = np.asarray(long_audio)

seconds = len(long_audio) / 16000
minutes, seconds = divmod(seconds, 60)
print(f"Length of audio sample is {minutes} minutes {seconds:.2f} seconds")

Length of audio sample is 5.0 minutes 20.19 seconds


chunk_length_s, controls the length of the chunked segments in seconds. For Whisper, 30 second chunks are optimal, since this matches the input length Whisper expects.

In [26]:
pipe(
    long_audio,
    max_new_tokens=256,
    generate_kwargs={"task":"transcribe", "language":"italian", "no_repeat_ngram_size":4},
    chunk_length_s=30,
    batch_size=8,
    return_timestamps=True)["chunks"] # whisper small has better performance on italian 

# "no_repeat_ngram_size": 4 - add to kwargs if repetitions still happens

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main

[{'timestamp': (0.0, 87.0),
  'text': " Certo, certo, cotto, cotto Asher, cotta, cotta Cotta, cota, cotta Cotta, palla, cotta per unacco e la cacco. C'è la cacca e la caca. C'è una cacca. Cacco e caca. A, A, di Passo, Ero da Tato, ero da Cio a Cio da Suspa Pava e Cio a Tato. andiamo a scegliere, andiamo allora a farlo. Non c'è un po' che c'è che non c'è una scelta, ma c'è la scelta."},
 {'timestamp': (87.0, 92.0),
  'text': " Non cè un po'. Ad assurre all'asciutto, ad all'assciutto."},
 {'timestamp': (92.0, 99.0), 'text': ' Ad assciutto, asciutto.'},
 {'timestamp': (99.0, 107.0), 'text': " Asciutto di all'asscia. Alla fine."},
 {'timestamp': (107.0, 108.0), 'text': ' Alla fine'},
 {'timestamp': (108.0, 109.0), 'text': ' Alla fine'},
 {'timestamp': (109.0, 110.0), 'text': ' Alla fine'},
 {'timestamp': (110.0, 111.0), 'text': ' Alla fine'},
 {'timestamp': (111.0, 112.0), 'text': ' Alla fine'},
 {'timestamp': (112.0, 113.0), 'text': ' Alla fine'},
 {'timestamp': (113.0, 114.0), 'text': ' 